In [1]:
import json
from kafka import KafkaConsumer
from dataclasses import dataclass

server = "localhost:9092"
topic_name = "healthcare_vitals"

In [2]:
@dataclass(slots=True)
class VitalEvent:
    subject_id: int
    device_id: int
    event_time: int
    heart_rate: int
    oxygen_level: int
    systolic_bp: int
    diastolic_bp: int
    body_temperature: float
    activity_level: str
    alert_flag: bool

In [3]:
def event_deserializer(v):
    event_json = v.decode("utf-8")
    event_dict = json.loads(event_json)
    return VitalEvent(**event_dict)

In [4]:
consumer = KafkaConsumer(
    topic_name,
    bootstrap_servers=[server],
    auto_offset_reset='earliest',
    group_id='health-database',
    value_deserializer=event_deserializer
)

In [15]:
record = next(consumer)

In [14]:
record.value

VitalEvent(subject_id=49169, device_id=10021, event_time=1776399585485, heart_rate=82, oxygen_level=97, systolic_bp=113, diastolic_bp=82, body_temperature=36.9, activity_level='resting', alert_flag=False)

In [5]:
for record in consumer:
    print(record.value)

VitalEvent(subject_id=21240, device_id=10017, event_time=1776400169948, heart_rate=122, oxygen_level=92, systolic_bp=134, diastolic_bp=81, body_temperature=37.3, activity_level='running', alert_flag=True)
VitalEvent(subject_id=42230, device_id=10016, event_time=1776400170148, heart_rate=94, oxygen_level=97, systolic_bp=124, diastolic_bp=88, body_temperature=36.5, activity_level='walking', alert_flag=False)
VitalEvent(subject_id=19318, device_id=10001, event_time=1776400170249, heart_rate=76, oxygen_level=95, systolic_bp=114, diastolic_bp=82, body_temperature=36.8, activity_level='resting', alert_flag=False)
VitalEvent(subject_id=21729, device_id=10023, event_time=1776400170350, heart_rate=127, oxygen_level=90, systolic_bp=140, diastolic_bp=89, body_temperature=37.7, activity_level='running', alert_flag=True)
VitalEvent(subject_id=13780, device_id=10009, event_time=1776400170451, heart_rate=84, oxygen_level=97, systolic_bp=122, diastolic_bp=78, body_temperature=36.6, activity_level='res

KeyboardInterrupt: 

In [7]:
import psycopg2

conn = psycopg2.connect(
    host='localhost',
    port=5433,
    database='postgres',
    user='postgres',
    password='postgres'
)
conn.autocommit = True
cur = conn.cursor()

In [8]:
{'subject_id': '20242',
 'device_id': '10023',
 'event_time': '2026-04-16T15:54:51.567660+00:00',
 'heart_rate': 68,
 'oxygen_level': 98,
 'systolic_bp': 176,
 'diastolic_bp': 108,
 'body_temperature': 36.7,
 'activity_level': 'resting',
 'alert_flag': True}

# Create table if not exists
cur.execute("""
CREATE TABLE IF NOT EXISTS vitals (
    subject_id VARCHAR(255),
    device_id VARCHAR(255),
    event_time TIMESTAMPTZ,
    heart_rate INT,
    oxygen_level INT,
    systolic_bp INT,
    diastolic_bp INT,
    body_temperature FLOAT,
    activity_level VARCHAR(255),
    alert_flag BOOLEAN
)
""")

In [9]:
for record in consumer:
    event = record.value
    print(event)
    cur.execute(
        """INSERT INTO vitals (
            subject_id, 
            device_id, 
            event_time, 
            heart_rate, 
            oxygen_level, 
            systolic_bp,
            diastolic_bp, 
            body_temperature, 
            activity_level, 
            alert_flag)
            VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s)""",
        (
            event.subject_id,
            event.device_id,
            event.event_time,
            event.heart_rate,
            event.oxygen_level,
            event.systolic_bp,
            event.diastolic_bp,
            event.body_temperature,
            event.activity_level,
            event.alert_flag,
        )
    )
    
consumer.close()
cur.close()
conn.close()

VitalEvent(subject_id='27974', device_id='10005', event_time='2026-04-16T16:22:52.767233+00:00', heart_rate=68, oxygen_level=97, systolic_bp=113, diastolic_bp=76, body_temperature=36.6, activity_level='resting', alert_flag=False)
VitalEvent(subject_id='26606', device_id='10019', event_time='2026-04-16T16:22:53.769204+00:00', heart_rate=79, oxygen_level=99, systolic_bp=111, diastolic_bp=71, body_temperature=36.5, activity_level='resting', alert_flag=False)
VitalEvent(subject_id='28919', device_id='10005', event_time='2026-04-16T16:22:54.771220+00:00', heart_rate=102, oxygen_level=99, systolic_bp=110, diastolic_bp=84, body_temperature=37.3, activity_level='walking', alert_flag=False)
VitalEvent(subject_id='27970', device_id='10013', event_time='2026-04-16T16:22:55.774344+00:00', heart_rate=81, oxygen_level=98, systolic_bp=114, diastolic_bp=71, body_temperature=37.0, activity_level='resting', alert_flag=False)


KeyboardInterrupt: 